In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set your raw data path
raw_path = r"D:\Projects\End-to-end projects\10. UPI Payments Intelligence\Data\Raw"
cleaned_path = r"D:\Projects\End-to-end projects\10. UPI Payments Intelligence\Data\Cleaned"

# Load all three datasets
df_txn = pd.read_csv(raw_path + r"\upi_transactions.csv")
df_users = pd.read_csv(raw_path + r"\upi_users.csv")
df_merchants = pd.read_csv(raw_path + r"\upi_merchants.csv")

print(f"Transactions loaded: {len(df_txn):,} rows x {df_txn.shape[1]} columns")
print(f"Users loaded: {len(df_users):,} rows x {df_users.shape[1]} columns")
print(f"Merchants loaded: {len(df_merchants):,} rows x {df_merchants.shape[1]} columns")

Transactions loaded: 500,000 rows x 30 columns
Users loaded: 10,000 rows x 10 columns
Merchants loaded: 2,000 rows x 6 columns


In [2]:
# ── Transactions ──────────────────────────────────────────
print("=" * 60)
print("TRANSACTIONS — BASIC INFO")
print("=" * 60)
print(df_txn.dtypes)
print(f"\nMissing Values:")
print(df_txn.isnull().sum())
print(f"\nDuplicate Rows: {df_txn.duplicated().sum()}")

# ── Users ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("USERS — BASIC INFO")
print("=" * 60)
print(df_users.dtypes)
print(f"\nMissing Values:")
print(df_users.isnull().sum())

# ── Merchants ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("MERCHANTS — BASIC INFO")
print("=" * 60)
print(df_merchants.dtypes)
print(f"\nMissing Values:")
print(df_merchants.isnull().sum())

TRANSACTIONS — BASIC INFO
txn_id                    object
txn_datetime              object
txn_date                  object
txn_month                  int64
txn_year                   int64
txn_hour                   int64
day_of_week               object
user_id                   object
user_city                 object
user_city_tier            object
user_age                   int64
user_gender               object
user_activity_level       object
merchant_id               object
merchant_name             object
merchant_category         object
merchant_city             object
platform_used             object
preferred_platform        object
is_preferred_platform      int64
payment_mode              object
device_type               object
amount                   float64
status                    object
unusual_hour_flag          int64
high_amount_flag           int64
round_amount_flag          int64
velocity_flag              int64
fraud_score                int64
is_fraud         

In [3]:
# ── Fix datetime columns ───────────────────────────────────
df_txn['txn_datetime'] = pd.to_datetime(df_txn['txn_datetime'])
df_txn['txn_date'] = pd.to_datetime(df_txn['txn_date'])
df_users['registration_date'] = pd.to_datetime(df_users['registration_date'])

# ── Fix categorical columns ────────────────────────────────
categorical_txn = [
    'day_of_week', 'user_city_tier', 'user_gender',
    'user_activity_level', 'merchant_category', 'platform_used',
    'preferred_platform', 'payment_mode', 'device_type', 'status'
]

for col in categorical_txn:
    df_txn[col] = df_txn[col].astype('category')

categorical_users = ['gender', 'city_tier', 'preferred_platform', 'activity_level', 'device_type']
for col in categorical_users:
    df_users[col] = df_users[col].astype('category')

categorical_merchants = ['merchant_category', 'merchant_city_tier']
for col in categorical_merchants:
    df_merchants[col] = df_merchants[col].astype('category')

# ── Order day of week correctly ────────────────────────────
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_txn['day_of_week'] = pd.Categorical(df_txn['day_of_week'], categories=day_order, ordered=True)

print("Data types fixed ✅")
print(f"\nTransaction dtypes:")
print(df_txn[['txn_datetime', 'txn_date', 'day_of_week', 'amount', 'status']].dtypes)

Data types fixed ✅

Transaction dtypes:
txn_datetime    datetime64[ns]
txn_date        datetime64[ns]
day_of_week           category
amount                 float64
status                category
dtype: object


In [4]:
# ── Time based features ────────────────────────────────────
df_txn['is_weekend'] = df_txn['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

df_txn['time_of_day'] = pd.cut(
    df_txn['txn_hour'],
    bins=[0, 6, 12, 17, 21, 24],
    labels=['Late Night', 'Morning', 'Afternoon', 'Evening', 'Night'],
    right=False
)

df_txn['is_festival_month'] = df_txn['txn_month'].isin([8, 9, 10, 11]).astype(int)

df_txn['quarter'] = df_txn['txn_datetime'].dt.quarter

# ── Amount based features ──────────────────────────────────
df_txn['amount_bucket'] = pd.cut(
    df_txn['amount'],
    bins=[0, 100, 500, 1000, 5000, 10000, float('inf')],
    labels=['Micro (<100)', 'Small (100-500)', 'Medium (500-1K)', 
            'Large (1K-5K)', 'Very Large (5K-10K)', 'Ultra (10K+)']
)

# ── Transaction success flag ───────────────────────────────
df_txn['is_success'] = (df_txn['status'] == 'Success').astype(int)
df_txn['is_failed'] = (df_txn['status'] == 'Failed').astype(int)

# ── Cross city transaction ─────────────────────────────────
df_txn['is_cross_city'] = (df_txn['user_city'] != df_txn['merchant_city']).astype(int)

# ── Age group ─────────────────────────────────────────────
df_txn['age_group'] = pd.cut(
    df_txn['user_age'],
    bins=[0, 25, 35, 45, 55, 100],
    labels=['Gen Z (18-25)', 'Millennial (26-35)', 'Gen X (36-45)', 
            'Senior (46-55)', 'Elder (55+)']
)

# ── User level aggregates ──────────────────────────────────
print("Calculating user level aggregates...")

user_agg = df_txn.groupby('user_id').agg(
    total_txns          = ('txn_id', 'count'),
    total_spend         = ('amount', 'sum'),
    avg_txn_amount      = ('amount', 'mean'),
    max_txn_amount      = ('amount', 'max'),
    unique_merchants    = ('merchant_id', 'nunique'),
    unique_categories   = ('merchant_category', 'nunique'),
    unique_platforms    = ('platform_used', 'nunique'),
    failed_txns         = ('is_failed', 'sum'),
    fraud_txns          = ('is_fraud', 'sum'),
    cross_city_txns     = ('is_cross_city', 'sum'),
    festival_txns       = ('is_festival_month', 'sum'),
).reset_index()

user_agg['failure_rate']        = (user_agg['failed_txns'] / user_agg['total_txns'] * 100).round(2)
user_agg['cross_city_rate']     = (user_agg['cross_city_txns'] / user_agg['total_txns'] * 100).round(2)
user_agg['festival_txn_rate']   = (user_agg['festival_txns'] / user_agg['total_txns'] * 100).round(2)
user_agg['avg_spend_per_day']   = (user_agg['total_spend'] / 730).round(2)

print(f"User aggregates calculated ✅")

# ── Merchant level aggregates ──────────────────────────────
print("Calculating merchant level aggregates...")

merchant_agg = df_txn.groupby('merchant_id').agg(
    total_txns          = ('txn_id', 'count'),
    total_revenue       = ('amount', 'sum'),
    avg_txn_amount      = ('amount', 'mean'),
    unique_users        = ('user_id', 'nunique'),
    failed_txns         = ('is_failed', 'sum'),
    fraud_txns          = ('is_fraud', 'sum'),
).reset_index()

merchant_agg['failure_rate']    = (merchant_agg['failed_txns'] / merchant_agg['total_txns'] * 100).round(2)
merchant_agg['fraud_rate']      = (merchant_agg['fraud_txns'] / merchant_agg['total_txns'] * 100).round(2)

print(f"Merchant aggregates calculated ✅")

print(f"\nNew columns added to transactions: {df_txn.shape[1]}")
print(f"\nSample new features:")
print(df_txn[['txn_id', 'time_of_day', 'amount_bucket', 'is_weekend', 
              'is_festival_month', 'is_cross_city', 'age_group']].head())

Calculating user level aggregates...
User aggregates calculated ✅
Calculating merchant level aggregates...
Merchant aggregates calculated ✅

New columns added to transactions: 39

Sample new features:
        txn_id time_of_day  amount_bucket  is_weekend  is_festival_month  \
0  TXN00000001   Afternoon  Large (1K-5K)           1                  0   
1  TXN00000002   Afternoon   Micro (<100)           1                  1   
2  TXN00000003     Morning  Large (1K-5K)           1                  0   
3  TXN00000004     Evening  Large (1K-5K)           0                  0   
4  TXN00000005     Morning  Large (1K-5K)           0                  0   

   is_cross_city           age_group  
0              1  Millennial (26-35)  
1              1      Senior (46-55)  
2              1       Gen Z (18-25)  
3              1       Gen X (36-45)  
4              1  Millennial (26-35)  


In [6]:
import os

cleaned_path = r"D:\Projects\End-to-end projects\10. UPI Payments Intelligence\Data\Cleaned"

# ── Save cleaned transactions ──────────────────────────────
df_txn.to_csv(os.path.join(cleaned_path, 'cleaned_transactions.csv'), index=False)
print(f"✅ Cleaned transactions saved: {len(df_txn):,} rows x {df_txn.shape[1]} columns")

# ── Save user aggregates ───────────────────────────────────
user_agg.to_csv(os.path.join(cleaned_path, 'user_aggregates.csv'), index=False)
print(f"✅ User aggregates saved: {len(user_agg):,} rows x {user_agg.shape[1]} columns")

# ── Save merchant aggregates ───────────────────────────────
merchant_agg.to_csv(os.path.join(cleaned_path, 'merchant_aggregates.csv'), index=False)
print(f"✅ Merchant aggregates saved: {len(merchant_agg):,} rows x {merchant_agg.shape[1]} columns")

# ── Save users and merchants cleaned ──────────────────────
df_users.to_csv(os.path.join(cleaned_path, 'cleaned_users.csv'), index=False)
print(f"✅ Cleaned users saved: {len(df_users):,} rows")

df_merchants.to_csv(os.path.join(cleaned_path, 'cleaned_merchants.csv'), index=False)
print(f"✅ Cleaned merchants saved: {len(df_merchants):,} rows")

# ── Final summary ──────────────────────────────────────────
print(f"\n{'='*50}")
print(f"PHASE 3 COMPLETE — DATA CLEANING SUMMARY")
print(f"{'='*50}")
print(f"Total transactions:        {len(df_txn):,}")
print(f"Total users:               {len(df_users):,}")
print(f"Total merchants:           {len(df_merchants):,}")
print(f"Date range:                {df_txn['txn_date'].min().date()} to {df_txn['txn_date'].max().date()}")
print(f"Total transaction value:   ₹{df_txn['amount'].sum():,.0f}")
print(f"Avg transaction amount:    ₹{df_txn['amount'].mean():,.0f}")
print(f"Success rate:              {df_txn['is_success'].mean()*100:.1f}%")
print(f"Fraud cases:               {df_txn['is_fraud'].sum():,} ({df_txn['is_fraud'].mean()*100:.2f}%)")
print(f"Missing values:            {df_txn.isnull().sum().sum()}")
print(f"Duplicate rows:            {df_txn.duplicated().sum()}")
print(f"\nAll files saved to Cleaned folder ✅")

✅ Cleaned transactions saved: 500,000 rows x 39 columns
✅ User aggregates saved: 10,000 rows x 16 columns
✅ Merchant aggregates saved: 2,000 rows x 9 columns
✅ Cleaned users saved: 10,000 rows
✅ Cleaned merchants saved: 2,000 rows

PHASE 3 COMPLETE — DATA CLEANING SUMMARY
Total transactions:        500,000
Total users:               10,000
Total merchants:           2,000
Date range:                2022-01-01 to 2023-12-31
Total transaction value:   ₹1,498,632,303
Avg transaction amount:    ₹2,997
Success rate:              91.0%
Fraud cases:               247 (0.05%)
Missing values:            0
Duplicate rows:            0

All files saved to Cleaned folder ✅
